# unbind-tuple-unpack — ex1: ARENA-style two-level destructure of rays into (ox, oy, oz, dx, dy, dz)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `unbind-tuple-unpack`. Running the final beacon cell reports progress against the `PyTorch: Unbind tuple-unpack` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Unbind tuple-unpack` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbind-tuple-unpack`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbind-tuple-unpack"
DD_SUBTOPIC = "PyTorch: Unbind tuple-unpack"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Unbind for tuple destructure — quick refresher

`x.unbind(dim=k)` returns a **Python tuple** of `x.shape[k]` view-tensors, each with axis `k` removed. The Pythonic move is to *destructure* that tuple directly into named variables:

```python
origin, direction = rays.unbind(dim=1)         # rays: (B, 2, 3)
ox, oy, oz        = origin.unbind(dim=-1)      # origin: (B, 3)
```

**Why this beats indexing.** `rays[:, 0]` and `rays[:, 1]` also work, but they hide the *count* and the *labels*. Destructuring into named variables tells the reader "this axis has exactly 2 elements, here are their meanings". Mis-spell a name → instant `NameError`; mis-index → silent semantic bug.

**It's a tuple, not a list.** That means it works with `*args` splat (`func(*rays.unbind(dim=0))`) and with sequence unpacking — but *not* with list mutation. If you need to modify the slices before restacking, wrap in `list(...)` first.

**Views, not copies.** Writes through the destructured tensors alias the source (just like slicing). Read-only consumers don't care, but if you intend to mutate, call `.clone()` on each slice.

### Exercise 1 — ARENA-style two-level destructure of rays into (ox, oy, oz, dx, dy, dz)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply two levels of `unbind` + tuple unpacking to a `(B, 2, 3)` ray batch, producing six named `(B,)` tensors `(ox, oy, oz, dx, dy, dz)` ready for analytic per-axis arithmetic.
> Keywords: unbind, destructure, ray-tracing, named-components
> ```

**KCs targeted:** `unbind-returns-python-tuple`, `unbind-two-level-destructure`

Implement `ex1_destructure_rays(rays)`.

- `rays` has shape `(B, 2, 3)` — row 0 of each `(2, 3)` block is the origin, row 1 is the direction.
- Return a `dict` with six keys: `'ox'`, `'oy'`, `'oz'`, `'dx'`, `'dy'`, `'dz'` — each a `(B,)` tensor.

**Required idiom (not an option).**
1. First level: `origin, direction = rays.unbind(dim=1)` — tuple destructure of the size-2 axis.
2. Second level: `ox, oy, oz = origin.unbind(dim=-1)` and `dx, dy, dz = direction.unbind(dim=-1)` — tuple destructure of the size-3 axis.

**Forbidden alternatives.** You may NOT use `rays[:, 0, 0]` / `rays[:, 1, 2]` / `select` / index arithmetic. The drill is about exercising `unbind` *as a destructure*, not as a slicer. (The test does not enforce this — it's pedagogical discipline. Look at the solution if you're unsure.)

**Why this is worth a whole drill.** Two-level unbind + named destructure is the ARENA ray-tracing idiom that makes the Moller-Trumbore solver readable. Without it, the next line is `t_hit = -rays[:, 0, 1] / rays[:, 1, 1]` — write-only code. With it: `t_hit = -oy / dy`.

In [ ]:
def ex1_destructure_rays(rays: Tensor) -> dict:
    origin, direction = rays.unbind(dim=1)
    ox, oy, oz = origin.unbind(dim=-1)
    dx, dy, dz = direction.unbind(dim=-1)
    return {'ox': ox, 'oy': oy, 'oz': oz, 'dx': dx, 'dy': dy, 'dz': dz}


<details><summary>Solution</summary>

```python
def ex1_destructure_rays(rays: Tensor) -> dict:
    origin, direction = rays.unbind(dim=1)
    ox, oy, oz = origin.unbind(dim=-1)
    dx, dy, dz = direction.unbind(dim=-1)
    return {'ox': ox, 'oy': oy, 'oz': oz, 'dx': dx, 'dy': dy, 'dz': dz}
```

**The two `unbind` calls do different jobs.** `rays.unbind(dim=1)` peels the 2-axis (origin vs direction). `origin.unbind(dim=-1)` peels the 3-axis (x vs y vs z components). Both *destructure* an axis of known small length into named tensors — no explicit loop, no indexing.

**Why named over indexed.** Compare the next line of an actual ARENA solver:
```python
# Named (this exercise):
t_hit = -oy / dy
# Indexed (no destructure):
t_hit = -rays[:, 0, 1] / rays[:, 1, 1]
```
The indexed form is correct but read-only — any reviewer has to stop and decode `[:, 0, 1]`. Named is self-documenting.

**Six aliases, one storage.** All six returned tensors are views into the same underlying storage as `rays`. Cheap to create, instantly garbage-collected when the dict goes out of scope. The storage-range assertion in the test verifies this — the address of `oy` lies strictly inside the byte range owned by `rays`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()